In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.experiment import prepare_environment

prepare_environment("../api_keys")

## Backend

In [3]:
import art

from openpipe.client import OpenPipe
from art.local import LocalBackend

op_client = OpenPipe()
print("OpenPipe client initialized")

backend = LocalBackend(in_process=True)
print("Local backend initialized")

KeyboardInterrupt: 

## Model

In [ ]:
from src.configs.art_model_config import configs as art_model_configs
from src.models import load_art_model, load_vllm_model

print("Available ART model configurations:")
print(art_model_configs.keys())

model_name = "unsloth/Qwen2.5-14B-Instruct"
# model_name = "unsloth/llama-3.1-8B-instruct"
project_name = "easy2hard_dac_v2"

In [ ]:
from art.dev import InternalModelConfig, EngineArgs, InitArgs


model, dir_config = await load_art_model(
    model_name=model_name,
    project_name=project_name,
    backend=backend,
)

In [ ]:
# load_vllm_model(
#     model_name=model_name,
#     server_port=8200,
#     gpu_id=1,
# )

## Data

In [ ]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [ ]:
from src.vllm_client import VllmClient, VllmRouter, ArtVLLMClient

inference_clients: list[VllmClient] = [ArtVLLMClient(model)]

vllm_server_ports = []
for port in vllm_server_ports:
    vllm_client = VllmClient(port=port, base_model=model_name)
    inference_clients.append(vllm_client)

vllm_router: VllmRouter = VllmRouter(vllm_clients=inference_clients)

## Training and Inference 

In [ ]:
from easy2hard.trainer import Easy2HardTrainer
from src.trainer import TrainingConfig, PromptConfig, StopCriteria
from src.configs.prompts import DaCSystemPrompt


train_config = TrainingConfig(
    epochs=10,
    num_groups=2,
    group_size=10,
    eval_steps=1,
    verbose=False,
    min_reward_stdev=None,
    art_config=art.types.TrainConfig(learning_rate=1e-5),
)

sys_prompt = PromptConfig(
    system_root=DaCSystemPrompt.dac_sys_prompt_gilad_root,
    system_inter=DaCSystemPrompt.dac_sys_prompt_gilad_inter,
    system_leaf=DaCSystemPrompt.dac_sys_prompt_gilad_leaf,
)

stop_criteria = StopCriteria(
    max_depth=1,
    max_tasks=5,
    max_rounds=5,
)

trainer = Easy2HardTrainer(
    model=model,
    client_list=inference_clients,
    dir_config=dir_config,
    train_config=train_config,
    prompt_config=sys_prompt,
    stop_criteria=stop_criteria,
)

### Inference Test

In [ ]:
import random

idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]
problem = sample["problem"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

preds = await trainer.predict([sample], verbose=True, seed=0)

### Train

In [ ]:
await trainer.train(
    train_dataset=train_data.to_list(),
    eval_dataset=test_data.to_list(),
)